# 01 — RAG multimodal: Qdrant + Gemini Embeddings + LLM en Ollama cloud

Primer notebook de una serie **incremental** de tres:

| Notebook | Qué construye |
|---|---|
| **01 (este)** | Un pipeline de **RAG multimodal** completo y evaluado |
| 02 | Un **agente** LangGraph que usa este RAG como *herramienta* |
| 03 | Los mismos componentes conectados vía **MCP** |

## El caso de uso

**TecnoMarket**, una tienda en línea ficticia, quiere un asistente que responda
preguntas usando su conocimiento interno:

- **Documentos de políticas** (envíos, devoluciones, garantías) — texto largo
  que hay que *trocear*.
- **Catálogo de productos** — cada producto tiene descripción **y fotografía**.

La gracia del caso es que es **multimodal**: el modelo de embeddings
`gemini-embedding-2` proyecta **texto e imágenes al MISMO espacio vectorial**,
así que una consulta escrita ("zapatillas de lona rojas") puede recuperar
directamente la *foto* del producto, sin pasos intermedios de anotación manual.

## La arquitectura

```
 políticas (.md) ──▶ chunking ──▶ embed (texto) ──┐
 descripciones  ─────────────────▶ embed (texto) ──┤──▶ ┌────────┐
 fotos (.jpg)   ─────────────────▶ embed (imagen) ─┘    │ Qdrant │  vectores + payloads
                                                        └───┬────┘
 pregunta ──▶ embed ──▶ búsqueda densa ─┐                   │
          └─▶ BM25 (léxica) ────────────┤──▶ fusión RRF ◀───┘
                                        ▼
                            contexto recuperado (texto + imágenes)
                                        ▼
                       prompt (LangChain) ──▶ LLM (Ollama cloud) ──▶ respuesta + fuentes
```

## Lo que cubrimos

1. **Embeddings multimodales** — un solo espacio para texto e imagen (Gemini).
2. **Qdrant** — desplegar la base vectorial y diseñar la colección.
3. **Chunking** — estrategia de troceado con LangChain.
4. **Ingesta** — indexar chunks, descripciones e imágenes.
5. **Recuperación** — densa, léxica (BM25) e **híbrida** (RRF).
6. **Generación** — cadena de LangChain con un LLM de Ollama cloud.
7. **Evaluación** — *hit rate* y *MRR*, explicados y medidos.
8. **Widgets** — explorar y validar los resultados interactivamente.

## Antes de ejecutar

```bash
cd module4-genai
uv sync                                   # dependencias
cp .env.example .env                      # y rellena GEMINI_API_KEY y OLLAMA_API_KEY
cd qdrant && docker compose up -d && cd . # desplegar Qdrant
```

- `GEMINI_API_KEY`: https://aistudio.google.com/apikey (los embeddings usan la API de Gemini).
- `OLLAMA_API_KEY`: https://ollama.com/settings/keys (el LLM corre en Ollama cloud).
- Dashboard de Qdrant: <http://localhost:6333/dashboard>

In [ ]:
# Carga la configuración desde module4-genai/.env (cópiala de .env.example).
import os
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd()
if not (ROOT / "rag").exists():          # si ejecutas desde notebooks/
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

GEMINI_EMBEDDING_MODEL = os.getenv("GEMINI_EMBEDDING_MODEL", "gemini-embedding-2")
EMBEDDING_DIM = int(os.getenv("EMBEDDING_DIM", "768"))
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "https://ollama.com")
QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
COLLECTION = os.getenv("QDRANT_COLLECTION", "tecnomarket")

print("Módulo:", ROOT)
print("Embeddings:", GEMINI_EMBEDDING_MODEL, f"({EMBEDDING_DIM} dim)")
print("Qdrant:", QDRANT_URL, "| colección:", COLLECTION)
print("GEMINI_API_KEY definida:", bool(os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")))
print("OLLAMA_API_KEY definida:", bool(os.getenv("OLLAMA_API_KEY")))

## 1. Los datos: catálogo y políticas de TecnoMarket

Los datos viven en `rag/catalog/`:

```
rag/catalog/
├── products.json    ~100 productos: sku, nombre, categoría, precio, descripción, imagen
├── docs/            3 políticas en Markdown: envíos, devoluciones, garantías
└── images/          ~100 fotografías de producto (JPEG, licencias CC)
```

Con ~100 productos en ~26 categorías el índice deja de ser de juguete: hay
**distractores** (varias zapatillas, varios relojes…), que es lo que hace
interesante medir la calidad de la recuperación en la sección 9.

In [ ]:
import base64
import json

import pandas as pd

CATALOG_DIR = ROOT / "rag" / "catalog"
PRODUCTS = json.loads((CATALOG_DIR / "products.json").read_text(encoding="utf-8"))
DOCS = {p.name: p.read_text(encoding="utf-8")
        for p in sorted((CATALOG_DIR / "docs").glob("*.md"))}

df_prod = pd.DataFrame(PRODUCTS)
print(f"{len(PRODUCTS)} productos en {df_prod['categoria'].nunique()} categorías, "
      f"{len(DOCS)} documentos de políticas: {list(DOCS)}")
print("\nProductos por categoría:")
print(df_prod["categoria"].value_counts().to_string())
df_prod[["sku", "nombre", "categoria", "precio", "imagen"]].head(8)

In [ ]:
# Mini-galería del catálogo. Incrustamos las imágenes como base64 para que el
# HTML sea autocontenido (mismo truco que usaremos al mostrar resultados).
from IPython.display import HTML, display

def img_b64(path):
    return "data:image/jpeg;base64," + base64.b64encode(Path(path).read_bytes()).decode()

def galeria(productos, ancho=140):
    tarjetas = []
    for p in productos:
        src = img_b64(CATALOG_DIR / "images" / p["imagen"])
        tarjetas.append(
            f'<div style="display:inline-block;margin:6px;text-align:center;width:{ancho}px;'
            f'vertical-align:top;font-family:sans-serif;font-size:11px">'
            f'<img src="{src}" style="width:{ancho}px;height:{ancho}px;object-fit:cover;'
            f'border-radius:8px"><br><b>{p["sku"]}</b><br>{p["nombre"]}</div>'
        )
    return HTML("<div>" + "".join(tarjetas) + "</div>")

display(galeria(PRODUCTS[:12]))
print(f"(mostrando 12 de {len(PRODUCTS)}; el resto se explora con los widgets de la sección 10)")

## 2. Embeddings multimodales con `gemini-embedding-2`

Un **embedding** convierte un contenido en un vector denso en $\mathbb{R}^d$,
de modo que contenidos semánticamente parecidos queden **cerca**. La medida de
cercanía que usamos es la **similitud coseno**:

$$\cos(\theta) = \frac{\mathbf{a} \cdot \mathbf{b}}{\lVert \mathbf{a}\rVert\,\lVert \mathbf{b}\rVert}$$

Lo nuevo aquí: `gemini-embedding-2` es **nativamente multimodal**. Texto,
imágenes, audio y PDF se proyectan al **mismo espacio vectorial**, así que
podemos comparar directamente una *consulta escrita* contra la *foto* de un
producto. Antes de estos modelos, el truco habitual era generar una descripción
de cada imagen con un modelo de visión y embeber esa descripción — un paso
extra, con pérdida de información.

Tres detalles prácticos del modelo:

- **Dimensiones flexibles (MRL)** — *Matryoshka Representation Learning*
  permite truncar el vector de 3072 a 1536 o **768** dims (nuestro caso: 4x
  menos almacenamiento con una pérdida de calidad mínima). El modelo
  **re-normaliza automáticamente** el vector truncado (norma 1).
- **Formato de la entrada** — para recuperación, la convención del modelo es
  formatear consulta y documento de forma distinta (asimétrica):
  - consulta: `task: search result | query: {texto}`
  - documento: `title: {título} | text: {texto}`
- **⚠ Agregación** — si pasas una **lista** en `contents`, el modelo devuelve
  **UN solo embedding agregado** de todo el conjunto (útil para "texto +
  imagen = un producto", traicionero si esperabas un embedding por elemento).
  Por eso embebemos **elemento por elemento**.

In [ ]:
import time
from functools import lru_cache

from google import genai
from google.genai import types

gclient = genai.Client()  # lee GEMINI_API_KEY / GOOGLE_API_KEY del entorno

def _embed(contents):
    # Reintentos con backoff exponencial: la API de Gemini tiene límites de
    # tasa (429) que en el tier gratuito se alcanzan rápido.
    for intento in range(5):
        try:
            r = gclient.models.embed_content(
                model=GEMINI_EMBEDDING_MODEL,
                contents=contents,
                config=types.EmbedContentConfig(output_dimensionality=EMBEDDING_DIM),
            )
            return list(r.embeddings[0].values)
        except Exception as exc:
            espera = 2 ** intento
            print(f"  reintento en {espera}s ({str(exc)[:80]})")
            time.sleep(espera)
    raise RuntimeError("La API de embeddings no respondió tras 5 intentos.")

@lru_cache(maxsize=512)
def embed_consulta(texto):
    # Cache en memoria: la evaluación (§9) y los widgets (§10) repiten las
    # mismas consultas — no tiene sentido pagar la API dos veces por ellas.
    return _embed(f"task: search result | query: {texto}")

def embed_documento(texto, titulo="none"):
    return _embed(f"title: {titulo} | text: {texto}")

def embed_imagen(path):
    part = types.Part.from_bytes(data=Path(path).read_bytes(), mime_type="image/jpeg")
    return _embed([part])

v = embed_consulta("zapatillas de lona rojas y azules")
print("Dimensión:", len(v), "| primeros 5 valores:", [round(x, 4) for x in v[:5]])

In [ ]:
# La prueba de fuego multimodal: consultas de TEXTO contra FOTOS de producto.
# Sin OCR, sin descripciones intermedias: coseno directo entre modalidades.
import numpy as np

consultas = [
    "zapatillas de lona rojas y azules",
    "una máquina para preparar café espresso",
    "bicicleta de montaña con doble suspensión",
]
imagenes = ["zapatillas-urbanas.jpg", "cafetera-espresso.jpg",
            "bicicleta-montana.jpg", "teclado-mecanico.jpg"]

vecs_img = {img: np.array(embed_imagen(CATALOG_DIR / "images" / img))
            for img in imagenes}

filas = []
for q in consultas:
    vq = np.array(embed_consulta(q))
    filas.append({img: round(float(vq @ v / (np.linalg.norm(vq) * np.linalg.norm(v))), 3)
                  for img, v in vecs_img.items()})

pd.DataFrame(filas, index=consultas)
# Cada fila debería tener su máximo en la imagen correcta.

## 3. Desplegar Qdrant y diseñar la colección

**Qdrant** es nuestra base de datos vectorial. La desplegamos con Docker
(`qdrant/docker-compose.yml` de este módulo):

```bash
cd qdrant && docker compose up -d
```

Comparar la consulta contra cada vector almacenado sería $O(N)$; Qdrant indexa
con **HNSW** (*Hierarchical Navigable Small World*), un grafo multicapa que da
búsquedas de vecinos casi logarítmicas a cambio de una pérdida mínima de
*recall* (por eso "ANN": *Approximate Nearest Neighbors*).

Diseño de nuestra colección:

- **Vectores**: 768 dims, distancia **coseno**.
- **Payload** (metadatos JSON por punto) — la clave del diseño multimodal es
  el campo `tipo`, que nos deja filtrar por modalidad:

| campo | valores | para qué |
|---|---|---|
| `tipo` | `doc` / `producto_texto` / `producto_imagen` | filtrar por modalidad |
| `ref` | `envios.md#2`, `TM-1001:texto`, `TM-1001:imagen` | identificar el punto (evaluación) |
| `texto` | el chunk o la descripción (vacío en imágenes) | armar el contexto del LLM |
| `sku`, `nombre`, `imagen`, `fuente` | metadatos del catálogo | mostrar fuentes / filtrar |

In [ ]:
from qdrant_client import QdrantClient, models

qdrant = QdrantClient(url=QDRANT_URL)
try:
    qdrant.get_collections()
except Exception as exc:
    raise RuntimeError(
        f"Qdrant no responde en {QDRANT_URL}. "
        "Inícialo con: cd qdrant && docker compose up -d"
    ) from exc

if qdrant.collection_exists(COLLECTION):
    qdrant.delete_collection(COLLECTION)
qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=models.VectorParams(size=EMBEDDING_DIM,
                                       distance=models.Distance.COSINE),
)
print(f"Colección '{COLLECTION}' creada ({EMBEDDING_DIM} dim, coseno).")

## 4. Estrategia de chunking (troceado) con LangChain

Los documentos largos no se embeben enteros. Dos razones:

1. **Precisión de la recuperación** — un vector que promedia 5 temas no queda
   cerca de ninguno. Chunks enfocados = vectores enfocados.
2. **Contexto del LLM** — recuperamos pasajes, no archivos: el prompt final
   debe ser pequeño y relevante.

Las decisiones de la estrategia:

- **Tamaño del chunk** — chico (200–400 chars): precisión alta, pero
  fragmentos sin contexto suficiente para el LLM. Grande (1500+): más
  contexto, pero vectores difusos y prompts caros. Para políticas cortas como
  las nuestras, un punto medio de **~700 caracteres** funciona bien.
- **Solapamiento (overlap)** — repetir el final de un chunk al inicio del
  siguiente (~15–20%) evita perder las ideas que cruzan la frontera.
- **Dónde cortar** — un corte a mitad de frase rompe el significado. El
  `RecursiveCharacterTextSplitter` de LangChain intenta separadores en orden
  (`\n\n` párrafos → `\n` líneas → `. ` frases → espacios) y solo baja al
  siguiente nivel si el fragmento sigue siendo demasiado grande.

Nuestros documentos además tienen **estructura Markdown** (secciones `##`), y
los límites de sección son límites semánticos naturales: cortar primero por
párrafos los respeta bastante bien.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

texto_envios = DOCS["envios.md"]

# Ingenuo: corta cada N caracteres, sin mirar dónde.
naive = CharacterTextSplitter(separator="", chunk_size=700, chunk_overlap=0)
# Recursivo: respeta párrafos > líneas > frases > palabras, con solapamiento.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700, chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""],
)

print("=== Corte ingenuo (fin del chunk 1) ===")
print("…" + naive.split_text(texto_envios)[0][-80:])
print("\n=== Corte recursivo (fin del chunk 1) ===")
print("…" + splitter.split_text(texto_envios)[0][-80:])

In [ ]:
# Troceamos los tres documentos de políticas. Cada chunk conserva su fuente y
# su posición: la referencia "envios.md#2" identifica al chunk 2 de envios.md.
CHUNKS = []
for fuente, texto in DOCS.items():
    for i, chunk in enumerate(splitter.split_text(texto)):
        CHUNKS.append({"ref": f"{fuente}#{i}", "fuente": fuente, "texto": chunk})

print(f"{len(CHUNKS)} chunks en total")
pd.DataFrame([{"ref": c["ref"], "caracteres": len(c["texto"]),
               "inicio": c["texto"][:60].replace("\n", " ") + "…"} for c in CHUNKS])

## 5. Ingesta multimodal

Indexamos **tres tipos de puntos** en la misma colección:

1. los **chunks** de las políticas (embedding de texto),
2. las **descripciones** de los productos (embedding de texto),
3. las **fotografías** de los productos (embedding de imagen).

Cada punto lleva un ID **determinista** (UUID5 derivado de su `ref`): si
re-ejecutas la ingesta, los puntos se sobreescriben en lugar de duplicarse.

> **Cache de embeddings en disco.** Con ~100 productos, la ingesta completa
> son **~215 llamadas** a la API (≈15 chunks + 100 descripciones + 100 fotos)
> — varios minutos con los límites del tier gratuito. Para no pagar eso en
> cada re-ejecución, cacheamos cada vector en
> `rag/catalog/embeddings_cache.npz`, **con clave = hash del contenido**: si
> el contenido no cambió, el vector sale del disco; si editas una descripción
> o cambias el modelo/dimensión, solo ese elemento vuelve a la API. El repo
> incluye el cache ya calculado, así que la primera ingesta también es rápida.

In [ ]:
import hashlib
import uuid

CACHE_PATH = CATALOG_DIR / "embeddings_cache.npz"
_cache = {}
if CACHE_PATH.exists():
    with np.load(CACHE_PATH) as z:
        _cache = {k: z[k] for k in z.files}
print(f"Cache de embeddings: {len(_cache)} vectores en disco")

_cache_nuevos = 0

def _clave(*partes):
    return hashlib.sha1("|".join(partes).encode()).hexdigest()

def _cacheado(clave, calcular):
    # Devuelve el vector del cache, o lo calcula (API) y lo agrega.
    global _cache_nuevos
    if clave not in _cache:
        _cache[clave] = np.asarray(calcular(), dtype=np.float32)
        _cache_nuevos += 1
    return _cache[clave].tolist()

def punto(ref, vector, payload):
    return models.PointStruct(
        id=str(uuid.uuid5(uuid.NAMESPACE_URL, ref)),
        vector=vector,
        payload={"ref": ref, **payload},
    )

BASE = (GEMINI_EMBEDDING_MODEL, str(EMBEDDING_DIM))
puntos = []

for c in CHUNKS:  # 1) chunks de políticas
    vec = _cacheado(_clave("doc", *BASE, c["fuente"], c["texto"]),
                    lambda c=c: embed_documento(c["texto"], titulo=c["fuente"]))
    puntos.append(punto(c["ref"], vec,
                        {"tipo": "doc", "fuente": c["fuente"], "texto": c["texto"]}))
print(f"✔ {len(CHUNKS)} chunks de políticas")

for i, p in enumerate(PRODUCTS):  # 2) descripciones + 3) fotografías
    vec_txt = _cacheado(_clave("texto", *BASE, p["nombre"], p["descripcion"]),
                        lambda p=p: embed_documento(p["descripcion"], titulo=p["nombre"]))
    puntos.append(punto(f"{p['sku']}:texto", vec_txt,
                        {"tipo": "producto_texto", "sku": p["sku"], "nombre": p["nombre"],
                         "imagen": p["imagen"], "texto": p["descripcion"]}))
    ruta_img = CATALOG_DIR / "images" / p["imagen"]
    vec_img = _cacheado(_clave("imagen", *BASE, hashlib.sha1(ruta_img.read_bytes()).hexdigest()),
                        lambda r=ruta_img: embed_imagen(r))
    puntos.append(punto(f"{p['sku']}:imagen", vec_img,
                        {"tipo": "producto_imagen", "sku": p["sku"], "nombre": p["nombre"],
                         "imagen": p["imagen"], "texto": ""}))
    if (i + 1) % 20 == 0:
        print(f"  … {i + 1}/{len(PRODUCTS)} productos")
print(f"✔ {len(PRODUCTS)} descripciones y {len(PRODUCTS)} imágenes")

if _cache_nuevos:
    np.savez_compressed(CACHE_PATH, **_cache)
    print(f"Cache actualizado: +{_cache_nuevos} vectores nuevos → {CACHE_PATH.name}")
else:
    print("Todo salió del cache: 0 llamadas a la API en la ingesta.")

qdrant.upsert(collection_name=COLLECTION, points=puntos)
print(f"\nInsertados {len(puntos)} puntos. Total en Qdrant:",
      qdrant.count(COLLECTION).count)

# Índices auxiliares en memoria que reutilizaremos:
PAYLOAD_POR_REF = {pt.payload["ref"]: pt.payload for pt in puntos}
CORPUS_TEXTO = [(ref, pl["texto"]) for ref, pl in PAYLOAD_POR_REF.items() if pl["texto"]]

## 6. Recuperación densa (semántica)

Embebemos la consulta (con su formato `task: search result | query: …`) y
pedimos a Qdrant los `top_k` vecinos por coseno. Gracias al campo `tipo` del
payload también podemos **filtrar por modalidad**.

In [ ]:
def buscar_por_vector(vector, top_k=5, tipo=None):
    # Núcleo de la búsqueda densa: recibe un VECTOR (da igual si vino de un
    # texto o de una imagen — mismo espacio) y consulta Qdrant.
    filtro = None
    if tipo is not None:
        filtro = models.Filter(must=[models.FieldCondition(
            key="tipo", match=models.MatchValue(value=tipo))])
    hits = qdrant.query_points(
        collection_name=COLLECTION, query=vector,
        limit=top_k, with_payload=True, query_filter=filtro,
    ).points
    return [{"score": round(h.score, 3), **h.payload} for h in hits]

def buscar_denso(consulta, top_k=5, tipo=None):
    return buscar_por_vector(embed_consulta(consulta), top_k=top_k, tipo=tipo)

def mostrar(resultados):
    piezas = []
    for i, r in enumerate(resultados, 1):
        cabeza = f'<b>[{i}]</b> <code>{r["ref"]}</code> · score {r["score"]} · <i>{r["tipo"]}</i>'
        cuerpo = (r["texto"][:180] + "…") if r["texto"] else f'<b>{r.get("nombre", "")}</b> (imagen)'
        thumb = ""
        if r.get("imagen"):
            thumb = (f'<img src="{img_b64(CATALOG_DIR / "images" / r["imagen"])}" '
                     f'style="width:72px;height:72px;object-fit:cover;border-radius:6px;'
                     f'margin-right:10px;float:left">')
        piezas.append(f'<div style="overflow:auto;margin:6px 0;font-family:sans-serif;'
                      f'font-size:12px">{thumb}{cabeza}<br>{cuerpo}</div>')
    return HTML("".join(piezas))

mostrar(buscar_denso("¿cuánto tarda el envío express?", top_k=3))

In [ ]:
# Consulta cross-modal: pedimos SOLO imágenes. El texto de la consulta se
# compara directamente contra los vectores de las fotos.
mostrar(buscar_denso("algo para escuchar música sin ruido de fondo",
                     top_k=3, tipo="producto_imagen"))

## 7. Búsqueda híbrida: densa + BM25, fusionadas con RRF

La búsqueda densa entiende **paráfrasis** ("algo para escuchar música" →
audífonos), pero puede tropezar con **términos exactos** poco frecuentes:
SKUs, códigos, nombres propios. Ahí brilla la búsqueda **léxica** clásica.

**BM25** puntúa un documento $d$ para una consulta $q$ sumando, por cada
término $t$ de la consulta:

$$\text{BM25}(d, q) = \sum_{t \in q} \text{IDF}(t)\cdot
\frac{f(t,d)\,(k_1+1)}{f(t,d) + k_1\left(1 - b + b\,\frac{|d|}{\overline{|d|}}\right)}$$

donde $f(t,d)$ es la frecuencia del término en el documento, $\text{IDF}$
castiga los términos comunes, y $k_1, b$ son constantes. La intuición: premia
coincidencias exactas de palabras raras, con saturación.

Para **fusionar** ambos rankings usamos *Reciprocal Rank Fusion* (RRF), que
combina **posiciones** (no puntuaciones, que viven en escalas incomparables):

$$\text{RRF}(d) = \sum_{r \in \text{retrievers}} \frac{1}{k + \text{rank}_r(d)}$$

con $k = 60$ típicamente (amortigua el peso de los primeros puestos).

> Detalle multimodal honesto: las **imágenes no tienen texto**, así que BM25
> no puede verlas — solo la rama densa las recupera. La fusión igual funciona:
> los puntos que solo aparecen en un ranking simplemente suman un término.

In [ ]:
from rank_bm25 import BM25Okapi

def _tokenizar(texto):
    return texto.lower().split()

_refs_bm25 = [ref for ref, _ in CORPUS_TEXTO]
_bm25 = BM25Okapi([_tokenizar(texto) for _, texto in CORPUS_TEXTO])

def buscar_bm25(consulta, top_k=5):
    scores = _bm25.get_scores(_tokenizar(consulta))
    orden = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [{"score": round(float(scores[i]), 3), **PAYLOAD_POR_REF[_refs_bm25[i]]}
            for i in orden if scores[i] > 0]

def buscar_hibrido(consulta, top_k=5, pool=15, rrf_k=60):
    denso = buscar_denso(consulta, top_k=pool)
    lexico = buscar_bm25(consulta, top_k=pool)
    fusion = {}
    for rank, r in enumerate(denso):
        fusion[r["ref"]] = fusion.get(r["ref"], 0) + 1.0 / (rrf_k + rank)
    for rank, r in enumerate(lexico):
        fusion[r["ref"]] = fusion.get(r["ref"], 0) + 1.0 / (rrf_k + rank)
    mejores = sorted(fusion, key=fusion.get, reverse=True)[:top_k]
    return [{"score": round(fusion[ref], 5), **PAYLOAD_POR_REF[ref]} for ref in mejores]

# El caso donde lo léxico gana: una consulta con SKU exacto.
q = "características de la bicicleta TM-1006"
print("— Solo denso:")
for r in buscar_denso(q, top_k=3):
    print(f"   [{r['score']}] {r['ref']}")
print("— Solo BM25:")
for r in buscar_bm25(q, top_k=3):
    print(f"   [{r['score']}] {r['ref']}")
print("— Híbrido (RRF):")
for r in buscar_hibrido(q, top_k=3):
    print(f"   [{r['score']}] {r['ref']}")

## 8. Generación: LangChain + LLM en Ollama cloud

Para la generación usamos un LLM servido por **Ollama cloud** — mismo
protocolo que un Ollama local, pero el modelo (de cientos de miles de millones
de parámetros) corre en los servidores de Ollama y nos autenticamos con una
API key.

> **El modelo se elige en la celda siguiente** (variable `OLLAMA_MODEL`), no
> en el `.env`: cambiarlo es parte del experimento. Ojo con la facturación:
> algunos modelos (p. ej. `kimi-k3:cloud`) son *extra usage* y requieren plan
> de pago con saldo; si el elegido no responde, la celda **degrada
> automáticamente** a `gpt-oss:120b-cloud` (plan gratuito). El pipeline es
> idéntico con cualquiera.

La integración con LangChain es `ChatOllama` (paquete `langchain-ollama`)
apuntando `base_url` a `https://ollama.com` con el header de autorización.

In [ ]:
# ⚙️ El modelo de Ollama cloud se elige AQUÍ, en el notebook.
#    Catálogo: https://ollama.com/search?c=cloud . Algunas opciones:
#      "minimax-m3:cloud"    razonador (thinking)
#      "kimi-k3:cloud"       multimodal (visión); se factura como "extra usage"
#      "gpt-oss:120b-cloud"  incluido en el plan gratuito
OLLAMA_MODEL = "minimax-m3:cloud"

# Si el modelo elegido no está disponible en tu plan (p. ej. HTTP 402 por saldo
# de extra usage en cero), caemos automáticamente al fallback del plan gratuito.
OLLAMA_FALLBACK = "gpt-oss:120b-cloud"

from langchain_ollama import ChatOllama
from ollama import Client as OllamaClient

OLLAMA_HEADERS = {"Authorization": "Bearer " + os.environ["OLLAMA_API_KEY"]}

def conectar_llm(**kwargs):
    # kwargs extra van directo a ChatOllama (p. ej. reasoning=True).
    probe = OllamaClient(host=OLLAMA_HOST, headers=OLLAMA_HEADERS)
    for modelo in [OLLAMA_MODEL, OLLAMA_FALLBACK]:
        try:
            probe.chat(model=modelo,
                       messages=[{"role": "user", "content": "ok"}],
                       options={"num_predict": 1})
        except Exception as exc:
            print(f"⚠ {modelo} no disponible: {str(exc)[:110]}")
            continue
        print("✔ Usando el modelo:", modelo)
        return ChatOllama(
            model=modelo,
            base_url=OLLAMA_HOST,
            client_kwargs={"headers": OLLAMA_HEADERS},
            temperature=0.1,
            **kwargs,
        )
    raise RuntimeError("Ningún modelo de Ollama cloud respondió; revisa OLLAMA_API_KEY.")

llm = conectar_llm()

### La cadena de RAG con LCEL

Componemos el pipeline con la sintaxis declarativa de LangChain (*LCEL*):

```
{contexto: recuperar, pregunta: passthrough} → prompt → llm → parser
```

El prompt instruye al modelo a responder **solo** con el contexto recuperado y
a citar las fuentes `[n]` — el mecanismo anti-alucinación básico de RAG.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

PROMPT_RAG = ChatPromptTemplate.from_messages([
    ("system",
     "Eres el asistente de la tienda TecnoMarket. Responde en español usando "
     "ÚNICAMENTE el contexto proporcionado. Cita las fuentes con su número "
     "entre corchetes, como [1]. Si el contexto no contiene la respuesta, "
     "dilo claramente y no inventes."),
    ("human", "Contexto:\n{contexto}\n\nPregunta: {pregunta}"),
])

def formatear_contexto(resultados):
    bloques = []
    for i, r in enumerate(resultados, 1):
        if r["tipo"] == "producto_imagen":
            bloques.append(f"[{i}] (foto del producto {r['sku']} — {r['nombre']})")
        else:
            origen = r.get("fuente") or f"{r['sku']} — {r['nombre']}"
            bloques.append(f"[{i}] (fuente: {origen})\n{r['texto']}")
    return "\n\n".join(bloques)

def recuperar(pregunta, top_k=4):
    return buscar_hibrido(pregunta, top_k=top_k)

cadena_rag = (
    {"contexto": lambda x: formatear_contexto(recuperar(x)),
     "pregunta": RunnablePassthrough()}
    | PROMPT_RAG
    | llm
    | StrOutputParser()
)

pregunta = "¿Puedo devolver unos audífonos si ya abrí el empaque?"
print(cadena_rag.invoke(pregunta))

In [ ]:
# La misma respuesta, pero mostrando también las fuentes recuperadas (con foto
# cuando la fuente es una imagen del catálogo) — así se audita un RAG.
def responder_con_fuentes(pregunta, top_k=4):
    fuentes = recuperar(pregunta, top_k=top_k)
    respuesta = (PROMPT_RAG | llm | StrOutputParser()).invoke(
        {"contexto": formatear_contexto(fuentes), "pregunta": pregunta})
    print(respuesta)
    display(mostrar(fuentes))

responder_con_fuentes("¿Qué garantía tiene la cafetera y qué no cubre?")

### Bonus multimodal: el LLM también puede *ver* la imagen recuperada

Si el modelo elegido tiene **visión nativa** (p. ej. `kimi-k3:cloud`), además
de citar la foto como fuente podemos adjuntarla al mensaje para que la
describa o la use al responder. (Con un modelo solo-texto — `minimax-m3`,
`gpt-oss` — la celda lo detecta y lo explica en lugar de fallar.)

In [ ]:
consulta_visual = "zapatillas de lona para uso diario"
foto = next(r for r in buscar_denso(consulta_visual, top_k=1, tipo="producto_imagen"))
print("Imagen recuperada:", foto["ref"], "—", foto["nombre"])

try:
    cliente_ollama = OllamaClient(host=OLLAMA_HOST, headers=OLLAMA_HEADERS)
    r = cliente_ollama.chat(
        model=llm.model,
        messages=[{
            "role": "user",
            "content": "Describe esta foto de producto en una frase de catálogo, en español.",
            "images": [base64.b64encode(
                (CATALOG_DIR / "images" / foto["imagen"]).read_bytes()).decode()],
        }],
    )
    print("\n", r["message"]["content"])
except Exception as exc:
    print(f"\n(El modelo '{llm.model}' no aceptó la imagen: {str(exc)[:100]}.\n"
          " Con un modelo de visión nativa (p. ej. kimi-k3:cloud) esta celda "
          "describe la foto.)")

## 9. Evaluar la recuperación: *hit rate* y *MRR*

En RAG, si la recuperación falla, la generación no puede salvarla: el LLM no
puede citar lo que nunca vio. Por eso la recuperación se evalúa **por
separado** del LLM, y con métricas propias.

### El vocabulario, pieza por pieza

- **Evalset**: una lista de consultas de prueba. Para cada una, anotamos de
  antemano qué elementos del índice **cuentan como respuesta correcta**. A esa
  anotación se le llama ***ground truth*** (la verdad de referencia).
- **Relevante**: un resultado que está en el ground truth de esa consulta. Es
  una etiqueta **binaria**: cuenta o no cuenta (no hay "medio relevante").
- **Ranking**: la lista ordenada que devuelve el buscador. La **posición**
  (o *rank*) se cuenta desde 1.
- **@k** ("*at* k"): solo miramos los **primeros $k$** resultados del ranking
  e ignoramos el resto. ¿Por qué truncar? Porque al LLM solo le pasamos los
  top-$k$ chunks — un relevante en la posición 47 **no existe** para el
  usuario ni para el prompt. La métrica debe mirar lo mismo que el sistema usa.

### ¿Qué cuenta como "relevante" en NUESTRO evalset?

Dos reglas, según el tipo de consulta:

1. **Consultas sobre políticas** ("¿cuánto cuesta el envío express?"):
   cualquier chunk del documento correcto es relevante — `envios.md#0`,
   `envios.md#1`, … Todos contienen contexto legítimo del tema.
2. **Consultas de producto** ("audífonos con cancelación de ruido"): con ~100
   productos ya no hay UNA respuesta correcta — cualquier audífono del
   catálogo sirve. Por eso marcamos como relevante **todo producto de la
   categoría correcta** (su `:texto` y su `:imagen`). Esto se llama relevancia
   a nivel de categoría.

### Hit Rate @ k — "¿encontró *algo* útil?"

Para **una** consulta: vale **1** si entre los primeros $k$ resultados hay al
menos un relevante, y **0** si no. Para el evalset completo, se promedia:

$$\text{HitRate@}k = \frac{1}{|Q|} \sum_{q \in Q} \mathbb{1}\left[\,\exists\, d \in \text{top-}k(q) : d \text{ es relevante}\,\right]$$

($\mathbb{1}[\cdot]$ vale 1 si la condición se cumple, 0 si no; $|Q|$ es el
número de consultas.) Un HitRate@5 de 0.80 se lee: *"en el 80% de las
consultas, el top-5 contenía al menos una respuesta útil"*. Es la métrica de
**cobertura**: ¿le dimos al LLM alguna oportunidad de responder bien?

Su punto ciego: **ignora la posición dentro del top-k**. Relevante en el
puesto 1 y relevante en el puesto 5 puntúan igual (1.0).

### MRR @ k — "¿qué tan *arriba* quedó?"

Primero, para **una** consulta, el ***reciprocal rank* (RR)**: el inverso de
la posición del **primer** relevante.

| posición del 1er relevante | RR = 1/posición |
|---|---|
| 1 | 1.00 |
| 2 | 0.50 |
| 3 | 0.33 |
| 5 | 0.20 |
| no aparece en el top-k | 0.00 |

El inverso hace que la penalización sea **no lineal**: caer del puesto 1 al 2
cuesta 0.5 puntos, pero del 4 al 5 apenas 0.05. Refleja cómo se consume un
ranking: lo primero pesa mucho más.

**MRR** (*Mean* Reciprocal Rank) es el promedio de los RR sobre el evalset:

$$\text{MRR@}k = \frac{1}{|Q|} \sum_{q \in Q} \frac{1}{\text{rank}_q}
\qquad \text{(término} = 0 \text{ si no hay relevante en el top-}k\text{)}$$

Un MRR@5 de 0.70 se lee: *"en promedio, el primer resultado útil aparece
entre el puesto 1 y el 2"*. Es la métrica de **posición**.

### Ejemplo trabajado completo (3 consultas, k=3)

| consulta | top-3 (✓ = relevante) | hit@3 | rank 1er ✓ | RR@3 |
|---|---|---|---|---|
| A | **✓** ✗ ✗ | 1 | 1 | 1.00 |
| B | ✗ ✗ **✓** | 1 | 3 | 0.33 |
| C | ✗ ✗ ✗ | 0 | — | 0.00 |

$$\text{HitRate@3} = \frac{1+1+0}{3} = 0.67 \qquad
\text{MRR@3} = \frac{1.00 + 0.33 + 0.00}{3} = 0.44$$

**Por qué se necesitan las dos:** un sistema puede encontrar casi siempre algo
relevante (HitRate alto) pero enterrado en el puesto 4–5 (MRR bajo): "funciona,
pero el prompt del LLM arranca con ruido". Otro puede ser preciso cuando
acierta (MRR alto por sus unos) pero fallar en muchas consultas. Mirar solo
una métrica esconde uno de los dos problemas.

Empecemos por ver la **anatomía de una sola consulta**, con el ranking real y
sus marcas ✓/✗ — y luego sí, el agregado.

In [ ]:
# Ground truth: relevancia por CATEGORÍA para productos, por DOCUMENTO para
# políticas. Un prefijo "TM-1042" marca TM-1042:texto y TM-1042:imagen;
# un prefijo "envios.md" marca envios.md#0, envios.md#1, …
def skus_de(categoria):
    return [p["sku"] for p in PRODUCTS if p["categoria"] == categoria]

def es_relevante(ref, prefijos):
    return any(ref.startswith(p) for p in prefijos)

def rank_primer_relevante(refs, prefijos):
    # Posición (1-indexada) del primer resultado relevante; None si no aparece.
    for i, ref in enumerate(refs, 1):
        if es_relevante(ref, prefijos):
            return i
    return None

# --- Anatomía de UNA consulta: el ranking crudo con sus marcas ---
consulta = "audífonos inalámbricos con cancelación de ruido"
prefijos = skus_de("audifonos")
print(f"Consulta:   {consulta!r}")
print(f"Relevantes: cualquier ref que empiece por {prefijos}\n")

k = 5
refs = [r["ref"] for r in buscar_hibrido(consulta, top_k=k)]
for pos, ref in enumerate(refs, 1):
    marca = "✓" if es_relevante(ref, prefijos) else "✗"
    print(f"  {pos}. {marca} {ref}")

rank = rank_primer_relevante(refs, prefijos)
print(f"\nhit@{k} = {1 if rank else 0}   (¿hay al menos un ✓ en el top-{k}?)")
print(f"rank del primer ✓ = {rank}   →   RR@{k} = " +
      (f"1/{rank} = {1/rank:.2f}" if rank else "0.00"))
print("\nEl evalset repite esto para todas las consultas y promedia.")

In [ ]:
EVALSET = [
    # (consulta, prefijos de refs relevantes)
    # — políticas: relevante = cualquier chunk del documento correcto
    ("¿Cuánto cuesta el envío express y en cuánto tiempo llega?", ["envios.md"]),
    ("¿Qué pasa si la transportadora no logra entregar mi paquete?", ["envios.md"]),
    ("¿Puedo devolver unos audífonos si ya abrí la caja?", ["devoluciones.md"]),
    ("¿Cómo inicio una devolución desde mi cuenta?", ["devoluciones.md"]),
    ("¿Cuántos meses de garantía tiene el teclado mecánico?", ["garantias.md"]),
    ("¿La garantía cubre el desgaste de la batería?", ["garantias.md"]),
    # — término exacto: aquí el único relevante SÍ es un producto concreto
    ("bicicleta TM-1006 precio y características", ["TM-1006"]),
    # — productos: relevante = cualquier producto de la categoría correcta
    ("zapatillas para correr con buena amortiguación", skus_de("zapatillas")),
    ("audífonos inalámbricos con cancelación de ruido", skus_de("audifonos")),
    ("una máquina para preparar café espresso en casa", skus_de("cafeteras")),
    ("mochila para hacer senderismo de varios días", skus_de("mochilas")),
    ("reloj de pulsera analógico clásico", skus_de("relojes")),
    ("un parlante para escuchar música en el parque", skus_de("altavoces")),
    ("una carpa impermeable para acampar el fin de semana", skus_de("carpas")),
    ("cámara para fotografía con controles manuales", skus_de("camaras")),
    ("silla cómoda para trabajar muchas horas frente al computador", skus_de("sillas")),
]

MODOS = {"denso": buscar_denso, "bm25": buscar_bm25, "hibrido": buscar_hibrido}

def evaluar(modo, k):
    buscar = MODOS[modo]
    hits, rr = [], []
    for consulta, prefijos in EVALSET:
        refs = [r["ref"] for r in buscar(consulta, top_k=k)]
        rank = rank_primer_relevante(refs, prefijos)
        hits.append(1.0 if rank else 0.0)
        rr.append(1.0 / rank if rank else 0.0)
    n = len(EVALSET)
    return {"hit_rate": round(sum(hits) / n, 3), "mrr": round(sum(rr) / n, 3)}

print("Evaluando", len(EVALSET), "consultas × 3 modos × k=1,3,5 …")
filas = []
for modo in MODOS:
    for k in (1, 3, 5):
        filas.append({"modo": modo, "k": k, **evaluar(modo, k)})
tabla = pd.DataFrame(filas).pivot(index="modo", columns="k",
                                  values=["hit_rate", "mrr"]).round(3)
tabla

### Cómo leer la tabla

- **Columnas**: la misma métrica calculada con distintos cortes $k$. La celda
  (`hibrido`, `hit_rate`, 3) responde: *"con búsqueda híbrida, ¿en qué
  fracción de las consultas hubo algo relevante en el top-3?"*.
- **`hibrido` domina o empata**: combina la comprensión semántica del denso
  con la precisión léxica de BM25. Es el default sensato para producción.
- **`bm25` sufre con las paráfrasis** — "algo para mantener el café caliente"
  no comparte palabras con "termo de acero inoxidable" — y es **ciego a las
  imágenes** (no tienen texto), así que su techo está limitado.
- **HitRate@k sube con $k$** por construcción (un corte más generoso da más
  oportunidades de capturar un relevante). El **MRR casi no cambia con $k$**:
  agrandar el corte agrega resultados *al final*, y el MRR solo depende de la
  posición del primer acierto (solo sube si un relevante que estaba fuera del
  corte entra en él).
- **En qué se diferencian denso e híbrido**: casi solo en la consulta con SKU
  exacto (`TM-1006`) — exactamente el caso que BM25 aporta.
- **Granularidad**: con 16 consultas, cada una mueve las métricas en
  1/16 ≈ 0.06 — diferencias menores a eso son ruido. En producción el evalset
  crece con consultas reales, y se re-ejecuta en CI ante cada cambio de
  chunking, modelo o parámetros: las métricas existen para detectar
  *regresiones*, no como nota final.
- **Límites de estas métricas**: relevancia binaria (no distingue "perfecto"
  de "aceptable") y solo miran el *primer* acierto. Si necesitas medir *cuántos*
  relevantes recuperas o con qué calidad de orden, las siguientes en la caja de
  herramientas son *recall@k* y *nDCG* — mismas ideas, contando más fino.

## 10. Validación interactiva con widgets

Tres paneles con `ipywidgets` para *tocar* el sistema:

1. **Búsqueda por texto** — consultas libres, cambiando modo (denso / BM25 /
   híbrido), `top_k` y filtro de modalidad.
2. **Búsqueda por imagen** — la consulta es una **foto** (una del catálogo o
   una que subas tú): se embebe con el mismo modelo y se busca por similitud
   en el mismo espacio vectorial. Sirve para probar imagen→imagen ("¿cuáles
   se parecen a esta?") e imagen→texto (filtra a "solo texto de productos" y
   verás que una foto encuentra su propia descripción).
3. **Panel de métricas** — recalcula hit rate y MRR del evalset al mover el
   modo y $k$, con el detalle por consulta: en qué posición quedó el primer
   relevante.

> Si los widgets no se muestran, habilita la extensión: en JupyterLab moderno
> `ipywidgets` funciona sin pasos extra; en VS Code usa el renderer de
> notebooks incluido.

In [ ]:
import ipywidgets as W

_q = W.Text(value="zapatillas rojas de lona", description="Consulta:",
            layout=W.Layout(width="450px"))
_modo = W.Dropdown(options=["hibrido", "denso", "bm25"], description="Modo:")
_k = W.IntSlider(value=4, min=1, max=10, description="top_k:")
_tipo = W.Dropdown(options=[("todos", None), ("solo políticas", "doc"),
                            ("solo texto de productos", "producto_texto"),
                            ("solo imágenes", "producto_imagen")],
                   description="Filtro:")
_boton = W.Button(description="Buscar", button_style="primary")
_salida = W.Output()

def _buscar_click(_):
    with _salida:
        _salida.clear_output()
        if _tipo.value is not None and _modo.value != "denso":
            print("(el filtro por tipo usa el índice de Qdrant → aplica en modo denso)")
            resultados = buscar_denso(_q.value, top_k=_k.value, tipo=_tipo.value)
        else:
            resultados = MODOS[_modo.value](_q.value, top_k=_k.value)
        if not resultados:
            print("Sin resultados.")
        else:
            display(mostrar(resultados))

_boton.on_click(_buscar_click)
display(W.VBox([W.HBox([_q, _boton]), W.HBox([_modo, _k, _tipo]), _salida]))
_buscar_click(None)  # primera búsqueda de ejemplo

In [ ]:
# --- Panel 2: búsqueda por IMAGEN (similitud visual) ---
# La consulta ya no es texto: es el embedding de una foto. Nota: si la foto
# elegida está en el índice, ella misma saldrá de primera con score ≈ 1.0 —
# un buen sanity check de que el espacio funciona.
_foto_dd = W.Dropdown(
    options=[(f'{p["sku"]} — {p["nombre"]}', p["imagen"]) for p in PRODUCTS],
    description="Foto:", layout=W.Layout(width="380px"))
_subida = W.FileUpload(accept="image/*", multiple=False, description="…o sube una")
_k_img = W.IntSlider(value=5, min=1, max=10, description="top_k:")
_tipo_img = W.Dropdown(options=[("todos", None), ("solo imágenes", "producto_imagen"),
                                ("solo texto de productos", "producto_texto"),
                                ("solo políticas", "doc")],
                       description="Filtro:")
_btn_img = W.Button(description="Buscar parecidos", button_style="primary")
_out_img = W.Output()

def _imagen_de_consulta():
    # Prioridad: archivo subido; si no hay, la foto elegida del catálogo.
    if _subida.value:
        item = (_subida.value[0] if isinstance(_subida.value, (list, tuple))
                else list(_subida.value.values())[0])         # ipywidgets 8 / 7
        data = bytes(item["content"])
        mime = item.get("type") or "image/jpeg"
        return data, mime, f"imagen subida ({item.get('name', '?')})"
    ruta = CATALOG_DIR / "images" / _foto_dd.value
    return ruta.read_bytes(), "image/jpeg", _foto_dd.value

def _buscar_imagen_click(_):
    with _out_img:
        _out_img.clear_output()
        data, mime, etiqueta = _imagen_de_consulta()
        display(HTML(
            f'<div style="font-family:sans-serif;font-size:12px">Consulta: '
            f'<b>{etiqueta}</b><br><img src="data:{mime};base64,'
            f'{base64.b64encode(data).decode()}" '
            f'style="width:110px;border-radius:8px;margin:4px 0"></div>'))
        vec = _embed([types.Part.from_bytes(data=data, mime_type=mime)])
        display(mostrar(buscar_por_vector(vec, top_k=_k_img.value, tipo=_tipo_img.value)))

_btn_img.on_click(_buscar_imagen_click)
display(W.VBox([W.HBox([_foto_dd, _subida]),
                W.HBox([_k_img, _tipo_img, _btn_img]), _out_img]))
_buscar_imagen_click(None)  # ejemplo inicial con la primera foto del catálogo

In [ ]:
# --- Panel 3: métricas en vivo — mueve modo y k, y mira el detalle por consulta.
_modo_m = W.Dropdown(options=["hibrido", "denso", "bm25"], description="Modo:")
_k_m = W.IntSlider(value=3, min=1, max=10, description="k:")
_salida_m = W.Output()

def _refrescar_metricas(*_):
    with _salida_m:
        _salida_m.clear_output()
        buscar = MODOS[_modo_m.value]
        detalle = []
        for consulta, prefijos in EVALSET:
            refs = [r["ref"] for r in buscar(consulta, top_k=_k_m.value)]
            rank = rank_primer_relevante(refs, prefijos)
            etiqueta = "|".join(prefijos[:3]) + ("…" if len(prefijos) > 3 else "")
            detalle.append({
                "consulta": consulta[:48] + ("…" if len(consulta) > 48 else ""),
                "relevantes": etiqueta,
                "rank_1er_relevante": rank if rank else "—",
                "recíproco": round(1.0 / rank, 3) if rank else 0.0,
            })
        df = pd.DataFrame(detalle)
        agg = evaluar(_modo_m.value, _k_m.value)
        print(f"modo={_modo_m.value}  k={_k_m.value}  →  "
              f"HitRate@{_k_m.value}={agg['hit_rate']}   MRR@{_k_m.value}={agg['mrr']}\n")
        display(df)

_modo_m.observe(_refrescar_metricas, names="value")
_k_m.observe(_refrescar_metricas, names="value")
display(W.VBox([W.HBox([_modo_m, _k_m]), _salida_m]))
_refrescar_metricas()

## Resumen

- `gemini-embedding-2` proyecta **texto e imágenes al mismo espacio**: la
  consulta escrita recupera fotos directamente (RAG multimodal sin
  anotaciones intermedias). Ojo con sus convenciones: formato asimétrico
  consulta/documento, y la **agregación** cuando `contents` es una lista.
- **Qdrant** guarda vectores + payloads; el campo `tipo` del payload nos dio
  filtrado por modalidad, y los IDs deterministas ingesta idempotente.
- El **chunking** es una decisión de diseño (tamaño, solapamiento, dónde
  cortar); `RecursiveCharacterTextSplitter` respeta las fronteras naturales.
- La búsqueda **híbrida** (densa + BM25 vía RRF) une paráfrasis y términos
  exactos; las imágenes solo son alcanzables por la rama densa.
- La generación es una cadena LCEL: recuperar → prompt con citas → `ChatOllama`
  (el modelo de Ollama cloud se elige en el notebook, con fallback automático).
- **Hit rate** (¿apareció algo relevante?) y **MRR** (¿qué tan arriba?) miden
  la recuperación sin tocar el LLM; los widgets permiten validar a mano lo que
  las métricas dicen en agregado.

**Siguiente** → `02_agentes_langgraph.ipynb`: convertimos este pipeline en una
*herramienta* que un agente LangGraph decide cuándo y cómo usar.